In [1]:
import os, sys, torch
from pathlib import Path
from dotenv import load_dotenv
import unsloth
from transformers import set_seed

load_dotenv()

SEED = 42
set_seed(SEED)

BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
sys.path.insert(0, str(ROOT_DIR))

CACHE_DIR = ROOT_DIR.parent / "huggingface_cache"
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATA_DIR = (ROOT_DIR / "data/pretrain_data/processed").resolve()
MODEL_DIR = (ROOT_DIR / "models/").resolve()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import unsloth
from unsloth import FastLanguageModel, get_chat_template

def get_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-8B",
        max_seq_length=2048,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")
    return model, tokenizer

In [3]:
model, tokenizer = get_model_and_tokenizer()

==((====))==  Unsloth 2026.4.2: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [4]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

PROMPT_3GPP = """3GPP Standard
### Title: {title}
### Text: {text}"""

PROMPT_ARXIV = """Arxiv
### Title: {title}
### Text: {text}"""

PROMPT_BOOKS = """Books
### Title: {title}
### Text: {text}"""

PROMPT_MAP = {
    "3gpp": PROMPT_3GPP,
    "arxiv": PROMPT_ARXIV,
    "books": PROMPT_BOOKS,
}

MAX_SEQ_LENGTH = 2048
CHUNK_STRIDE   = 256

In [5]:
def _chunk_batch(batch, tokenizer):
    all_texts, all_domains = [], []
    EOS = tokenizer.eos_token
    for title, raw_text, domain in zip(batch["title"], batch["raw_text"], batch["domain"]):
        PROMPT = PROMPT_MAP.get(domain, PROMPT_ARXIV)
        tokenized = tokenizer(
            raw_text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            return_overflowing_tokens=True,
            stride=CHUNK_STRIDE,
            add_special_tokens=False,
        )
        for ids in tokenized["input_ids"]:
            chunk_body = tokenizer.decode(ids, skip_special_tokens=True)
            all_texts.append(PROMPT.format(title=title, text=chunk_body) + EOS)
            all_domains.append(domain)
    return {"text": all_texts, "domain": all_domains}


def load_all_pretrain_data(root_dir: str, tokenizer, seed=42):
    root = Path(root_dir)
    datasets_list = []

    for file in (root / "3GPP").glob("*.jsonl"):
        release_name = file.stem.replace("Rel-", "Release ")
        ds = load_dataset("json", data_files=str(file), split="train")
        ds = ds.map(
            lambda x: {"title": release_name, "raw_text": x["text"], "domain": "3gpp"},
            remove_columns=ds.column_names,
        )
        datasets_list.append(ds)

    rp_file = next((root / "redpajama").glob("*.jsonl"))
    ds_rp = load_dataset("json", data_files=str(rp_file), split="train")
    ds_rp = ds_rp.map(
        lambda x: {"title": x["file_name"], "raw_text": x["text"], "domain": "arxiv"},
        remove_columns=ds_rp.column_names,
    )
    datasets_list.append(ds_rp)

    book_file = next((root / "books").glob("*.jsonl"))
    ds_book = load_dataset("json", data_files=str(book_file), split="train")
    ds_book = ds_book.map(
        lambda x: {"title": x["file_name"], "raw_text": x["text"], "domain": "arxiv"},
        remove_columns=ds_book.column_names,
    )
    datasets_list.append(ds_book)

    full_dataset = concatenate_datasets(datasets_list)

    full_dataset = full_dataset.map(
        lambda batch: _chunk_batch(batch, tokenizer),
        batched=True,
        batch_size=64,
        remove_columns=full_dataset.column_names,
        num_proc=1,
        desc="Chunking documents",
    )

    full_dataset = full_dataset.shuffle(seed=seed)
    return full_dataset


In [6]:
from datasets import ClassLabel

def stratified_split(dataset, val_ratio=0.05, seed=42):
    from datasets import ClassLabel, DatasetDict

    domain_names = sorted(set(dataset["domain"]))
    dataset = dataset.cast_column("domain", ClassLabel(names=domain_names))

    split_dataset = dataset.train_test_split(
        test_size=val_ratio,
        seed=seed,
        stratify_by_column="domain"
    )

    return DatasetDict({
        "train": split_dataset["train"],
        "validation": split_dataset["test"]
    })

In [7]:
dataset = load_all_pretrain_data(DATA_DIR, tokenizer)
dataset = stratified_split(dataset, val_ratio=0.1)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['domain', 'text'],
        num_rows: 68462
    })
    validation: Dataset({
        features: ['domain', 'text'],
        num_rows: 7607
    })
})


In [8]:
# import pandas as pd

# def count_tokens(example):
#     return {"length": len(tokenizer(example["text"])["input_ids"])}

# def get_stats(split):
#     length_ds = dataset[split].map(count_tokens, num_proc=4)
#     df = pd.DataFrame({
#         "domain": length_ds["domain"],
#         "tokens": length_ds["length"],
#     })
#     stats = df.groupby("domain")["tokens"].agg(
#         ["count", "sum", "mean", "max"]
#     ).reset_index()
#     stats.columns = [
#         "domain",
#         f"{split}_num_samples",
#         f"{split}_total_tokens",
#         f"{split}_avg_tokens",
#         f"{split}_max_tokens",
#     ]
#     return stats

# train_stats = get_stats("train")
# val_stats   = get_stats("validation")

# final_stats = train_stats.merge(val_stats, on="domain", how="outer")

# final_stats["overall_total_tokens"] = (
#     final_stats["train_total_tokens"] +
#     final_stats["validation_total_tokens"]
# )

# final_stats["val_token_ratio"] = (
#     final_stats["validation_total_tokens"] /
#     final_stats["overall_total_tokens"]
# )

# final_stats.sort_values("domain")

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = True,
    loftq_config = None,
)

Unsloth 2026.4.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [10]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 8,

    packing = False,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 4,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,

        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 2e-4,

        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 300,

        save_strategy = "steps",
        save_steps = 300,
        save_total_limit = 3,
        load_best_model_at_end = True,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = SEED,

        output_dir = MODEL_DIR,
        report_to = "wandb",

        gradient_checkpointing = True,
    ),
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [22]:
trainer_stats = trainer.train()
trainer.evaluate()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,462 | Num Epochs = 1 | Total steps = 4,279
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 349,175,808 of 8,539,911,168 (4.09% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [21]:
import os

prefix = "/opt/conda/envs/dopn-venv"

os.environ["PATH"] = prefix + "/bin:" + os.environ["PATH"]
os.environ["CC"] = prefix + "/bin/x86_64-conda-linux-gnu-gcc"
os.environ["CXX"] = prefix + "/bin/x86_64-conda-linux-gnu-g++"

os.environ["GCC_EXEC_PREFIX"] = prefix + "/lib/gcc/"

In [15]:
import os
os.environ["CC"] = "/opt/conda/envs/dopn-venv/libexec/gcc/x86_64-conda-linux-gnu/15.2.0/gcc"

import shutil
triton_cache = os.path.expanduser("~/.triton/cache")
if os.path.exists(triton_cache):
    shutil.rmtree(triton_cache)
    print("Cleared triton cache")

# Verify
import subprocess
result = subprocess.run(
    ["/opt/conda/envs/dopn-venv/libexec/gcc/x86_64-conda-linux-gnu/15.2.0/gcc", "--version"],
    capture_output=True, text=True
)
print(result.stdout)

Cleared triton cache
gcc (conda-forge gcc 15.2.0-18) 15.2.0
Copyright (C) 2025 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.


